# 01 — Exploratory Data Analysis
**Dataset:** UCI Individual Household Electric Power Consumption (Dec 2006 – Nov 2010)  
**Resolution:** 1 minute | ~2.1 M rows | 7 numeric sensor features

Sections:
1. Time-series overview (daily resample)  
2. Distributions — histograms + boxplots  
3. Correlation matrix  
4. Data quality (missing values, implausible readings)  
5. Seasonal patterns (hour-of-day, day-of-week, month, hour×DOW heatmap)

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (14, 4),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='darkgrid', palette='muted')

CLEAN_PATH = Path('..') / 'data' / 'processed' / 'clean.parquet'

COLS = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3',
]

In [ ]:
df = pd.read_parquet(CLEAN_PATH)
print(f'Shape          : {df.shape}')
print(f'Date range     : {df.index.min()}  →  {df.index.max()}')
print(f'Memory (MB)    : {df.memory_usage(deep=True).sum() / 1_048_576:.1f}')
print()
df.dtypes

## 1. Time-Series Overview (daily mean)
1-minute resolution is too dense to plot directly; daily resampling reveals trends and seasonality.

In [ ]:
daily = df[COLS].resample('D').mean()

fig, axes = plt.subplots(len(COLS), 1, figsize=(14, 3 * len(COLS)), sharex=True)
for ax, col in zip(axes, COLS):
    ax.plot(daily.index, daily[col], linewidth=0.8)
    ax.set_ylabel(col.replace('_', '\n'), fontsize=8)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[-1].tick_params(axis='x', rotation=30)
fig.suptitle('All 7 features — daily mean (Dec 2006 – Nov 2010)', y=1.01, fontsize=14)
fig.tight_layout()
plt.show()

## 2. Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes_flat = axes.flatten()

for i, col in enumerate(COLS):
    axes_flat[i].hist(df[col].dropna(), bins=80, edgecolor='none', alpha=0.75)
    axes_flat[i].set_title(col.replace('_', ' '), fontsize=9)
    axes_flat[i].set_xlabel('Value')
    axes_flat[i].set_ylabel('Count')

axes_flat[-1].set_visible(False)  # 7 features, 8 subplot slots
fig.suptitle('Histograms — 1-minute readings (NaN excluded)')
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df[COLS].boxplot(
    ax=ax,
    whis=1.5,
    flierprops={'marker': '.', 'markersize': 1, 'alpha': 0.2, 'color': 'steelblue'},
    medianprops={'color': 'tomato', 'linewidth': 2},
)
ax.set_xticklabels([c.replace('_', '\n') for c in COLS], fontsize=9)
ax.set_title('Boxplots — IQR fences at 1.5 × IQR; dots = outliers')
plt.tight_layout()
plt.show()

## 3. Correlation Matrix
Strong correlations between features are expected (active power ↔ current) and will be visible here.

In [ ]:
corr = df[COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
)
ax.set_title('Pearson correlation — lower triangle')
plt.tight_layout()
plt.show()

print('Pairs with |r| > 0.7:')
corr_long = (
    corr.where(~mask)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'A', 'level_1': 'B', 0: 'r'})
)
strong = corr_long[corr_long['r'].abs() > 0.7].sort_values('r', ascending=False)
print(strong.to_string(index=False))

## 4. Data Quality
### 4a. Missing values
Forward-fill (limit=5 min) was applied in the cleaning step.  NaN here means a gap > 5 minutes.

In [ ]:
total_rows = len(df)
na_counts = df[COLS].isna().sum()
na_pct = (na_counts / total_rows * 100).round(3)

quality_df = pd.DataFrame({'missing_rows': na_counts, 'pct_%': na_pct})
print(f'Total rows (1-min grid): {total_rows:,}')
print()
print(quality_df)

# Timeline of missing-minute clusters (weekly aggregation)
gap_flag = df['Global_active_power'].isna().astype(int)
weekly_gaps = gap_flag.resample('W').sum()

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(weekly_gaps.index, weekly_gaps.values, width=5, color='tomato', alpha=0.7)
ax.set_title('Missing minutes per week (long-gap NaN, Global_active_power)')
ax.set_ylabel('Minutes missing')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 4b. Implausible readings
Negative power/current readings and zero voltage are physiically impossible and indicate sensor faults.

In [ ]:
print('Negative values per column (sensor fault indicator):')
found_any = False
for col in COLS:
    n_neg = (df[col] < 0).sum()
    if n_neg:
        print(f'  {col}: {n_neg:,}')
        found_any = True
if not found_any:
    print('  None — all readings are non-negative.')

print()
zero_v = (df['Voltage'] == 0).sum()
print(f'Zero-voltage readings: {zero_v:,}')

print()
print('Summary statistics:')
df[COLS].describe().round(4)

## 5. Seasonal Patterns
Using `Global_active_power` as the representative feature.

In [ ]:
hourly_mean = df['Global_active_power'].groupby(df.index.hour).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly_mean.index, hourly_mean.values, color='steelblue', alpha=0.85)
ax.set_xticks(range(24))
ax.set_xlabel('Hour of day')
ax.set_ylabel('Mean active power (kW)')
ax.set_title('Average consumption by hour of day')
plt.tight_layout()
plt.show()

In [ ]:
dow_mean = df['Global_active_power'].groupby(df.index.dayofweek).mean()
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(day_labels, dow_mean.values, color='mediumseagreen', alpha=0.85)
ax.set_xlabel('Day of week')
ax.set_ylabel('Mean active power (kW)')
ax.set_title('Average consumption by day of week')
plt.tight_layout()
plt.show()

In [ ]:
month_mean = df['Global_active_power'].groupby(df.index.month).mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_labels, month_mean.values, color='coral', alpha=0.85)
ax.set_xlabel('Month')
ax.set_ylabel('Mean active power (kW)')
ax.set_title('Average consumption by month — clear winter heating effect')
plt.tight_layout()
plt.show()

In [ ]:
# Hour × day-of-week heatmap reveals morning/evening peaks on weekdays vs weekends
pivot = (
    df['Global_active_power']
    .groupby([df.index.dayofweek, df.index.hour])
    .mean()
    .unstack(level=1)
)
pivot.index = day_labels

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot, cmap='YlOrRd', linewidths=0.1,
    ax=ax, cbar_kws={'label': 'Mean kW'},
)
ax.set_title('Mean active power — hour of day × day of week')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Day of week')
plt.tight_layout()
plt.show()

## Summary of Key Findings

| Observation | Implication for modelling |
|---|---|
| Strong winter peak in monthly averages | Month/season features needed |
| Morning (~8h) and evening (~19h) daily peaks | Hour-of-day feature essential |
| Weekend vs weekday difference in DOW plot | Day-of-week dummy or cyclical encoding |
| High correlation: `Global_active_power` ↔ `Global_intensity` | Multicollinearity — consider dropping one in linear models |
| Missing-minute clusters are sparse and random | Remaining NaN rows can be dropped before training |
| Sub-metering 1/2 are near-zero most of the time | Right-skewed; log transform may help |

**Next step:** Phase 2 — feature engineering (rolling statistics, lag features, calendar encodings).